# 序号25：因子构建与检验

## 学习目标
- 独立构建估值、动量、质量三类因子
- 掌握因子预处理：去极值、标准化、正交化
- 理解并计算 **IC（Information Coefficient）** 和 **IR（Information Ratio）**
- 分析因子衰减现象和拥挤风险

## 验收标准
- ✅ 能解释 IC 和 IR 的含义
- ✅ 理解因子衰减现象
- ✅ 知道因子拥挤的风险

In [ ]:
# === 核心库 ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr, pearsonr
import akshare as ak
import warnings
warnings.filterwarnings('ignore')

# 中文字体
plt.rcParams['font.sans-serif'] = ['PingFang SC', 'Heiti SC', 'STHeiti', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 5)

print('✅ 环境就绪')

## 1. 数据准备

获取沪深300成分股近2年的日线数据，作为因子构建的底层数据。

In [ ]:
# 获取沪深300成分股列表
print('📡 获取沪深300成分股...')
hs300 = ak.index_stock_cons_csindex(symbol="000300")
stocks = hs300['成分券代码'].tolist()
print(f'沪深300成分股数量: {len(stocks)}')
print(f'前10只: {stocks[:10]}')

# 为了演示效率和可复现性，取前50只（按市值排序的前50已覆盖大市值代表）
sample_stocks = stocks[:50]
print(f'\n演示用股票池: {len(sample_stocks)}只 (前50大市值)')

In [ ]:
# 获取日线数据（取最近2年）
import time

price_data = {}
failed = []

print('📡 批量获取日线数据 (预计1-2分钟)...')
for i, code in enumerate(sample_stocks):
    try:
        df = ak.stock_zh_a_hist(symbol=code, period='daily', 
                                start_date='20240601', end_date='20260620',
                                adjust='qfq')
        if len(df) > 100:  # 至少100个交易日
            df['日期'] = pd.to_datetime(df['日期'])
            df = df.set_index('日期').sort_index()
            price_data[code] = df['收盘']
        else:
            failed.append(code)
    except Exception as e:
        failed.append(code)
    
    if (i+1) % 10 == 0:
        print(f'  进度: {i+1}/{len(sample_stocks)}')
    time.sleep(0.15)  # 控制请求频率

print(f'\n✅ 成功: {len(price_data)}只, ❌ 失败: {len(failed)}只')

# 构建价格DataFrame
price_df = pd.DataFrame(price_data)
print(f'价格数据形状: {price_df.shape}')
print(f'日期范围: {price_df.index[0].date()} ~ {price_df.index[-1].date()}')
price_df.tail(3).round(2)

## 2. 因子构建

### 2.1 估值因子（Value Factor）

估值因子的核心逻辑：**便宜的公司未来收益更高**。常用指标：
- **E/P (Earnings Yield)**：归母净利润 / 总市值，即 PE 的倒数
- **B/P (Book-to-Price)**：净资产 / 总市值，即 PB 的倒数
- **S/P (Sales-to-Price)**：营业收入 / 总市值

> 我们使用 E/P 和 B/P 的等权组合作为综合估值因子。

In [ ]:
# 获取财务数据构建估值因子
print('📡 获取财务指标...')

# 使用akshare获取估值数据
# 方法：批量获取个股PE、PB
valuation_data = {}
for i, code in enumerate(sample_stocks):
    try:
        # 获取个股实时估值
        info = ak.stock_individual_info_em(symbol=code)
        if info is not None and len(info) > 0:
            info_dict = dict(zip(info['item'], info['value']))
            valuation_data[code] = info_dict
    except:
        pass
    time.sleep(0.1)

print(f'获取到 {len(valuation_data)} 只股票的估值数据')

# 提取PE和PB
pe_dict, pb_dict = {}, {}
for code, info in valuation_data.items():
    try:
        pe = float(info.get('市盈率-动态', np.nan))
        pb = float(info.get('市净率', np.nan))
        if pe > 0 and pe < 200:
            pe_dict[code] = 1.0 / pe  # E/P
        if pb > 0 and pb < 20:
            pb_dict[code] = 1.0 / pb  # B/P
    except:
        pass

ep_series = pd.Series(pe_dict, name='EP')
bp_series = pd.Series(pb_dict, name='BP')

print(f'E/P因子: {len(ep_series)}只, 范围 [{ep_series.min():.4f}, {ep_series.max():.4f}]')
print(f'B/P因子: {len(bp_series)}只, 范围 [{bp_series.min():.4f}, {bp_series.max():.4f}]')

# 综合估值因子 = (E/P + B/P) / 2 (标准化后等权)
from scipy.stats import zscore

value_factor = (zscore(ep_series.dropna()) + zscore(bp_series.dropna())) / 2
value_factor.name = 'value'
value_factor = value_factor.dropna()
print(f'\n综合估值因子: {len(value_factor)}只股票, 范围 [{value_factor.min():.2f}, {value_factor.max():.2f}]')
value_factor.describe().round(4)

### 2.2 动量因子（Momentum Factor）

动量因子基于 **Jegadeesh & Titman (1993)** 的发现：过去表现好的股票未来也倾向于表现好。

我们构建两个动量因子：
- **12-1月动量**：过去12个月（跳过最近1个月）的累计收益
- **短期反转**：过去1个月的收益（通常为负相关，即反转效应）

In [ ]:
# 构建动量因子
# 12-1月动量: 从t-12到t-1的累计收益

# 计算月度收益
monthly_ret = price_df.resample('ME').last().pct_change()

# 12-1动量 (跳过最近1个月)
momentum_12m1 = {}
short_reversal = {}

for col in price_df.columns:
    try:
        ret = monthly_ret[col].dropna()
        if len(ret) >= 12:
            # 12-1月动量: 过去12个月（去掉最近1月）的累计收益
            momentum_12m1[col] = (1 + ret.iloc[-13:-1]).prod() - 1
            # 短期反转: 最近1个月收益
            short_reversal[col] = ret.iloc[-1]
    except:
        pass

mom_series = pd.Series(momentum_12m1, name='momentum_12m1')
rev_series = pd.Series(short_reversal, name='short_reversal')

print(f'12-1月动量因子: {len(mom_series)}只, 范围 [{mom_series.min():.2%}, {mom_series.max():.2%}]')
print(f'短期反转因子: {len(rev_series)}只, 范围 [{rev_series.min():.2%}, {rev_series.max():.2%}]')

# 综合动量因子 = 标准化后等权
momentum_factor = (zscore(mom_series.dropna()) + zscore(rev_series.dropna())) / 2
momentum_factor.name = 'momentum'
print(f'\n综合动量因子: {len(momentum_factor)}只')
momentum_factor.describe().round(4)

### 2.3 质量因子（Quality Factor）

质量因子捕捉公司基本面优秀的特征。常用指标：
- **ROE**：净资产收益率
- **毛利率**：毛利润/营业收入
- **资产负债率**：越低越好（负向指标）

In [ ]:
# 构建质量因子
print('📡 获取财务数据构建质量因子...')

roe_dict, gross_margin_dict, debt_ratio_dict = {}, {}, {}

for i, code in enumerate(sample_stocks):
    try:
        # 获取主要财务指标
        fin = ak.stock_financial_abstract_ths(symbol=code, indicator='按报告期')
        if fin is not None and len(fin) > 0:
            # ROE
            roe_row = fin[fin['报告期'].str.contains('净资产收益率', na=False)]
            if len(roe_row) > 0:
                roe_dict[code] = float(roe_row.iloc[0].iloc[1])
            
            # 毛利率
            gross_row = fin[fin['报告期'].str.contains('销售毛利率', na=False)]
            if len(gross_row) > 0:
                gross_margin_dict[code] = float(gross_row.iloc[0].iloc[1])
            
            # 资产负债率
            debt_row = fin[fin['报告期'].str.contains('资产负债率', na=False)]
            if len(debt_row) > 0:
                debt_ratio_dict[code] = float(debt_row.iloc[0].iloc[1])
    except Exception as e:
        pass
    
    if (i+1) % 20 == 0:
        print(f'  进度: {i+1}/{len(sample_stocks)}')
    time.sleep(0.3)

print(f'ROE: {len(roe_dict)}只, 毛利率: {len(gross_margin_dict)}只, 资产负债率: {len(debt_ratio_dict)}只')

roe_s = pd.Series(roe_dict, name='roe')
gross_s = pd.Series(gross_margin_dict, name='gross_margin')
debt_s = pd.Series(debt_ratio_dict, name='debt_ratio')

# 资产负债率越低越好，取负值
quality_factor = (zscore(roe_s.dropna()) + zscore(gross_s.dropna()) + zscore(-debt_s.dropna())) / 3
quality_factor.name = 'quality'
print(f'\n综合质量因子: {len(quality_factor)}只, 范围 [{quality_factor.min():.2f}, {quality_factor.max():.2f}]')
quality_factor.describe().round(4)

## 3. 因子预处理

### 3.1 因子对齐
将所有因子合并到统一股票池，处理缺失值。

In [ ]:
# 合并所有因子
factor_df = pd.DataFrame({
    'value': value_factor,
    'momentum': momentum_factor,
    'quality': quality_factor
}).dropna()

print(f'合并后因子数据: {factor_df.shape}')
print(f'共同股票池: {len(factor_df)}只')
factor_df.head(10).round(4)

### 3.2 去极值（Winsorization）

极端值会严重干扰因子回测结果。我们使用 **MAD法（Median Absolute Deviation）** 进行去极值处理：

$$\tilde{x} = \text{median}(x)$$
$$\text{MAD} = \text{median}(|x_i - \tilde{x}|)$$

将超过 $\tilde{x} \pm 5 \times \text{MAD}$ 的值拉回到边界。

In [ ]:
def winsorize_mad(series, n=5):
    """MAD法去极值"""
    median = series.median()
    mad = np.median(np.abs(series - median))
    if mad == 0:
        return series
    upper = median + n * mad
    lower = median - n * mad
    return series.clip(lower, upper)

# 去极值前统计
print('===== 去极值前 =====')
print(factor_df.describe().round(4))

# 应用去极值
factor_winsor = factor_df.apply(winsorize_mad)

print('\n===== 去极值后 =====')
print(factor_winsor.describe().round(4))

# 可视化对比
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, col in enumerate(factor_df.columns):
    axes[i].hist(factor_df[col], bins=30, alpha=0.5, label='原始', color='steelblue')
    axes[i].hist(factor_winsor[col], bins=30, alpha=0.5, label='去极值', color='coral')
    axes[i].set_title(col)
    axes[i].legend(fontsize=8)
plt.suptitle('去极值前后对比', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 3.3 标准化与正交化

**标准化**：将因子转为 Z-score，均值为0，标准差为1。

**正交化**：因子之间往往存在相关性（如低估值的股票往往质量也高），需要去除共线性。我们使用 **施密特正交化（Gram-Schmidt）** 或简单地对目标因子做残差化处理。

In [ ]:
# 标准化
factor_std = factor_winsor.apply(zscore)

print('===== 标准化后 =====')
print(f'均值:\n{factor_std.mean().round(6)}\n')
print(f'标准差:\n{factor_std.std().round(6)}\n')

# 因子相关性（正交化前）
corr_before = factor_std.corr()
print('===== 因子相关性（正交化前）=====')
print(corr_before.round(4))

In [ ]:
# 正交化：对每个因子用其他因子回归，取残差
# 这种方法称为"对称正交化"，保留因子经济含义

def sym_orthogonalize(factor_df):
    """对称正交化：每个因子对其他所有因子回归，取残差"""
    result = pd.DataFrame(index=factor_df.index)
    for col in factor_df.columns:
        y = factor_df[col]
        X = factor_df.drop(columns=[col])
        X = sm.add_constant(X)
        model = sm.OLS(y, X).fit()
        result[col] = model.resid
    return result

import statsmodels.api as sm

factor_orth = sym_orthogonalize(factor_std)

# 再标准化
factor_orth = factor_orth.apply(zscore)

corr_after = factor_orth.corr()
print('===== 因子相关性（正交化后）=====')
print(corr_after.round(4))

# 可视化对比
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(corr_before, annot=True, cmap='RdBu_r', center=0, vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title('正交化前')
sns.heatmap(corr_after, annot=True, cmap='RdBu_r', center=0, vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title('正交化后（对称正交化）')
plt.tight_layout()
plt.show()

## 4. IC（Information Coefficient）分析

### 4.1 什么是 IC？

**IC（信息系统）** 衡量因子值与未来收益之间的相关性：

$$IC_t = \text{corr}(\text{Factor}_{t}, \text{Return}_{t+1})$$

- **Pearson IC（Normal IC）**：因子值与下一期收益的 Pearson 相关系数
- **Rank IC（Spearman IC）**：因子排名与下一期收益排名的 Spearman 秩相关系数（更稳健）

| IC 绝对值 | 评价 |
|-----------|------|
| > 0.05 | 优秀因子 |
| 0.03-0.05 | 良好因子 |
| 0.02-0.03 | 可用因子 |
| < 0.02 | 弱因子 |

In [ ]:
# 计算未来收益（日度 → 月度）
# 取最近一个月的收益作为目标

# 获取最新价格日期
latest_date = price_df.index[-1]
one_month_ago = latest_date - pd.DateOffset(months=1)

# 未来1个月收益（简化：取最近一个月的实际收益作为"未来收益"的代理）
fwd_returns = {}
for col in price_df.columns:
    if col in factor_orth.index:
        try:
            # 如果有未来数据，用未来数据
            # 这里用最近1个月收益作为代理
            ret_1m = price_df[col].iloc[-1] / price_df[col].iloc[-22] - 1 if len(price_df[col]) >= 22 else np.nan
            fwd_returns[col] = ret_1m
        except:
            fwd_returns[col] = np.nan

fwd_ret_series = pd.Series(fwd_returns).dropna()
print(f'未来收益数据: {len(fwd_ret_series)}只')
print(f'收益范围: [{fwd_ret_series.min():.2%}, {fwd_ret_series.max():.2%}]')

In [ ]:
# 计算 Rank IC（Spearman）
ic_results = {}

for factor_name in factor_orth.columns:
    common_idx = factor_orth.index.intersection(fwd_ret_series.index)
    f_values = factor_orth.loc[common_idx, factor_name]
    f_returns = fwd_ret_series.loc[common_idx]
    
    # Rank IC
    rank_ic, rank_pval = spearmanr(f_values, f_returns)
    # Normal IC
    normal_ic, normal_pval = pearsonr(f_values, f_returns)
    
    ic_results[factor_name] = {
        'Rank_IC': rank_ic,
        'Rank_IC_pval': rank_pval,
        'Normal_IC': normal_ic,
        'Normal_IC_pval': normal_pval
    }

ic_df = pd.DataFrame(ic_results).T
print('===== IC 计算结果 =====')
print(ic_df.round(4))

# 解读
print('\n===== IC 解读 =====')
for factor_name, row in ic_df.iterrows():
    rank_ic = row['Rank_IC']
    sig = '显著' if row['Rank_IC_pval'] < 0.05 else '不显著'
    if abs(rank_ic) > 0.05:
        quality = '🌟 优秀'
    elif abs(rank_ic) > 0.03:
        quality = '✅ 良好'
    elif abs(rank_ic) > 0.02:
        quality = '⚠️ 可用'
    else:
        quality = '❌ 弱因子'
    direction = '正向' if rank_ic > 0 else '反向'
    print(f'{factor_name}: Rank_IC={rank_ic:.4f} ({quality}, {direction}, {sig})')

In [ ]:
# IC 可视化
fig, ax = plt.subplots(figsize=(10, 5))

colors = ['#2563eb' if v > 0 else '#dc2626' for v in ic_df['Rank_IC']]
bars = ax.bar(ic_df.index, ic_df['Rank_IC'], color=colors, alpha=0.8, edgecolor='white')

# 添加阈值线
ax.axhline(y=0.05, color='green', linestyle='--', alpha=0.5, label='优秀阈值 (0.05)')
ax.axhline(y=0.03, color='orange', linestyle='--', alpha=0.5, label='良好阈值 (0.03)')
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax.axhline(y=-0.05, color='green', linestyle='--', alpha=0.5)
ax.axhline(y=-0.03, color='orange', linestyle='--', alpha=0.5)

# 标注
for bar, (_, row) in zip(bars, ic_df.iterrows()):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., 
            height + 0.003 * np.sign(height),
            f'{row["Rank_IC"]:.4f}', 
            ha='center', va='bottom' if height > 0 else 'top', fontsize=11, fontweight='bold')

ax.set_title('因子 Rank IC 对比', fontsize=14, fontweight='bold')
ax.set_ylabel('Rank IC')
ax.legend(loc='upper right', fontsize=9)
ax.set_ylim(-0.1, max(ic_df['Rank_IC'].max() + 0.03, 0.08))
plt.tight_layout()
plt.show()

## 5. IC 衰减分析

### 5.1 什么是因子衰减？

**因子衰减（Factor Decay）** 是指因子的预测能力随时间推移而下降的现象。

- **原因**：市场学习效应、因子拥挤、因子逻辑失效
- **表现**：IC 随持有期增加而递减
- **应对**：缩短调仓周期、因子轮动、复合因子

我们计算不同持有期（1天、5天、10天、20天）下的 IC，观察衰减曲线。

In [ ]:
# 模拟IC衰减：计算不同持有期下的Rank IC
# 使用滚动窗口计算IC序列

# 计算各持有期的收益
holding_periods = [1, 5, 10, 20]  # 天

# 使用更完整的数据：计算过去多个截面的IC序列
ic_decay = {factor: [] for factor in factor_orth.columns}

# 取最近20个交易日，计算每5天一个截面的IC
n_slices = 6  # 最近6个截面

for slice_idx in range(n_slices):
    offset = slice_idx * 5
    if len(price_df) < offset + 22:
        break
    
    # 当期因子（用最新因子，简化处理：因子值假设不变）
    for holding in holding_periods:
        # 计算该持有期的未来收益
        fwd_ret_slice = {}
        ref_idx = -(1 + offset)
        fwd_idx = ref_idx - holding
        
        for col in price_df.columns:
            if col in factor_orth.index and len(price_df[col]) > abs(fwd_idx):
                try:
                    ret = price_df[col].iloc[fwd_idx] / price_df[col].iloc[ref_idx] - 1
                    fwd_ret_slice[col] = ret
                except:
                    pass
        
        fwd_ret_s = pd.Series(fwd_ret_slice).dropna()
        if len(fwd_ret_s) < 10:
            continue
        
        for factor_name in factor_orth.columns:
            common_idx = factor_orth.index.intersection(fwd_ret_s.index)
            if len(common_idx) < 10:
                continue
            f_vals = factor_orth.loc[common_idx, factor_name]
            f_rets = fwd_ret_s.loc[common_idx]
            ic, _ = spearmanr(f_vals, f_rets)
            ic_decay[factor_name].append({'holding': holding, 'ic': ic, 'slice': slice_idx})

# 整理为DataFrame
decay_records = []
for factor_name, records in ic_decay.items():
    for r in records:
        decay_records.append({
            'factor': factor_name,
            'holding_days': r['holding'],
            'ic': r['ic'],
            'slice': r['slice']
        })

decay_df = pd.DataFrame(decay_records)
# 按持有期聚合
decay_agg = decay_df.groupby(['factor', 'holding_days'])['ic'].mean().unstack(0)

print('===== IC衰减表（各持有期平均Rank IC）=====')
print(decay_agg.round(4))

In [ ]:
# 绘制IC衰减曲线
fig, ax = plt.subplots(figsize=(10, 5))

colors = {'value': '#2563eb', 'momentum': '#dc2626', 'quality': '#16a34a'}
markers = {'value': 'o', 'momentum': 's', 'quality': '^'}

for factor in decay_agg.columns:
    ax.plot(decay_agg.index, decay_agg[factor], 
            color=colors.get(factor, 'gray'), marker=markers.get(factor, 'o'),
            linewidth=2, markersize=8, label=factor, alpha=0.85)

ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax.set_xlabel('持有期（天）', fontsize=12)
ax.set_ylabel('平均 Rank IC', fontsize=12)
ax.set_title('IC 衰减曲线：因子预测能力随持有期变化', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# 添加拐点标注
for factor in decay_agg.columns:
    best_holding = decay_agg[factor].abs().idxmax()
    best_ic = decay_agg[factor].loc[best_holding]
    ax.annotate(f'{best_holding}天: {best_ic:.4f}',
                xy=(best_holding, best_ic),
                xytext=(best_holding + 2, best_ic + 0.005),
                fontsize=9, color=colors.get(factor, 'gray'),
                arrowprops=dict(arrowstyle='->', color=colors.get(factor, 'gray'), alpha=0.6))

plt.tight_layout()
plt.show()

In [ ]:
# IC 半衰期：IC 降到初始值一半所需的持有期
print('===== IC 半衰期分析 =====')
for factor in decay_agg.columns:
    series = decay_agg[factor]
    initial_ic = series.iloc[0]
    half_ic = initial_ic / 2
    if initial_ic > 0:
        # 找到第一个低于半衰期的持有期
        below_half = series[series < half_ic]
        if len(below_half) > 0:
            half_life = below_half.index[0]
        else:
            half_life = f'>{series.index[-1]}天'
    else:
        half_life = 'N/A (IC为负)'
    
    print(f'{factor}: 初始IC={initial_ic:.4f}, 半衰IC={half_ic:.4f}, 半衰期≈{half_life}')

## 6. IR（Information Ratio）分析

### 6.1 什么是 IR？

**信息比率（IR）** 衡量因子的风险调整后预测能力：

$$IR = \frac{\text{mean}(IC_t)}{\text{std}(IC_t)}$$

- **含义**：因子预测能力的一致性。IR 越高，因子表现越稳定
- **与 IC 的区别**：IC 衡量平均预测力，IR 衡量预测力的稳定性
- **经验法则**：IR > 0.5 为优秀，IR > 0.3 为良好

$$\text{年化 IR} = IR \times \sqrt{\text{调仓频率}}$$

In [ ]:
# 计算 IC 序列的均值和标准差 → IR
print('===== IR 分析 =====')

ir_results = {}
for factor_name in ic_decay:
    recs = ic_decay[factor_name]
    # 取持有期为1的IC序列（日度IC）
    ic_series_1d = [r['ic'] for r in recs if r['holding'] == 1]
    if len(ic_series_1d) >= 3:
        ic_mean = np.mean(ic_series_1d)
        ic_std = np.std(ic_series_1d, ddof=1)
        ir = ic_mean / ic_std if ic_std > 0 else 0
        # 年化IR：日度IR * sqrt(252)
        ir_annual = ir * np.sqrt(252)
        ir_results[factor_name] = {
            'IC_mean': ic_mean,
            'IC_std': ic_std,
            'IR_daily': ir,
            'IR_annual': ir_annual
        }

ir_df = pd.DataFrame(ir_results).T
print(ir_df.round(4))

print('\n===== IR 解读 =====')
for factor_name, row in ir_df.iterrows():
    ir_ann = row['IR_annual']
    if abs(ir_ann) > 0.5:
        quality = '🌟 优秀'
    elif abs(ir_ann) > 0.3:
        quality = '✅ 良好'
    else:
        quality = '⚠️ 一般'
    consistency = '高一致性' if abs(row['IC_std']) < 0.05 else '波动较大'
    print(f'{factor_name}: 年化IR={ir_ann:.4f} ({quality}), IC波动={row["IC_std"]:.4f} ({consistency})')

In [ ]:
# IR 可视化
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 左: IC均值 vs IC波动率散点图 (IR = 斜率)
ax = axes[0]
for factor_name, row in ir_df.iterrows():
    ax.scatter(row['IC_std'], row['IC_mean'], s=200, 
              color=colors.get(factor_name, 'gray'),
              edgecolors='white', linewidth=2, zorder=5)
    ax.annotate(factor_name, (row['IC_std'], row['IC_mean']),
                textcoords='offset points', xytext=(10, 5), fontsize=11)

# IR=0.5 参考线
x_vals = np.linspace(0, ir_df['IC_std'].max() * 1.2, 50)
ax.plot(x_vals, 0.5 * x_vals, '--', color='green', alpha=0.5, label='IR=0.5 (优秀)')
ax.plot(x_vals, 0.3 * x_vals, '--', color='orange', alpha=0.5, label='IR=0.3 (良好)')
ax.axhline(y=0, color='gray', alpha=0.3)
ax.set_xlabel('IC 标准差', fontsize=12)
ax.set_ylabel('IC 均值', fontsize=12)
ax.set_title('IR 散点图：IC均值 vs IC标准差', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 右: 年化IR柱状图
ax = axes[1]
bars = ax.bar(ir_df.index, ir_df['IR_annual'], 
              color=[colors.get(f, '#gray') for f in ir_df.index], alpha=0.8)
ax.axhline(y=0.5, color='green', linestyle='--', alpha=0.5, label='优秀')
ax.axhline(y=0.3, color='orange', linestyle='--', alpha=0.5, label='良好')
for bar, (_, row) in zip(bars, ir_df.iterrows()):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., h + 0.02 * np.sign(h),
            f'{row["IR_annual"]:.3f}', ha='center', fontsize=11, fontweight='bold')
ax.set_title('年化 IR 对比', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. 因子稳定性检验

### 7.1 滚动 IC 分析

因子稳定性的核心指标：
- **IC > 0 的比例**：应在 50% 以上
- **IC 的 t 统计量**：|t| > 2 表示显著不等于 0
- **IC 自相关**：低自相关说明因子有持续预测力（而非偶然）

In [ ]:
# IC > 0 比例和 t 统计量
print('===== 因子稳定性检验 =====')

stability_results = {}
for factor_name in ic_decay:
    ic_vals = [r['ic'] for r in ic_decay[factor_name] if r['holding'] == 1]
    if len(ic_vals) < 3:
        continue
    
    ic_arr = np.array(ic_vals)
    n = len(ic_arr)
    
    # IC > 0 比例
    pos_ratio = np.mean(ic_arr > 0)
    
    # t统计量
    t_stat = np.mean(ic_arr) / (np.std(ic_arr, ddof=1) / np.sqrt(n)) if np.std(ic_arr, ddof=1) > 0 else 0
    p_value = 2 * (1 - stats.t.cdf(abs(t_stat), n-1))
    
    # 胜率
    win_rate = pos_ratio
    
    stability_results[factor_name] = {
        'n_periods': n,
        'IC_pos_ratio': pos_ratio,
        't_statistic': t_stat,
        'p_value': p_value,
        'significant': p_value < 0.05
    }

stability_df = pd.DataFrame(stability_results).T
print(stability_df.round(4))

print('\n===== 稳定性解读 =====')
for factor_name, row in stability_df.iterrows():
    sig = '✅ 显著' if row['significant'] else '❌ 不显著'
    consistency = '🟢 稳定' if row['IC_pos_ratio'] > 0.6 else ('🟡 一般' if row['IC_pos_ratio'] > 0.5 else '🔴 不稳定')
    print(f'{factor_name}: IC>0比例={row["IC_pos_ratio"]:.0%} ({consistency}), t={row["t_statistic"]:.2f} ({sig})')

## 8. 因子拥挤风险

### 8.1 什么是因子拥挤？

**因子拥挤（Factor Crowding）** 指过多资金追捧同一因子，导致：
1. **因子估值泡沫**：因子内股票价格被推高到脱离基本面
2. **回撤风险加剧**：拥挤解除时发生踩踏（如 2007 年 Quant Quake）
3. **因子失效**：超额收益被套利殆尽

**识别信号**：
- 因子估值（多空组合的估值价差处于历史高位）
- 因子内个股相关性异常升高
- 因子波动率突然下降（"波动率黑洞"）
- 主动基金对该因子的暴露集中度上升

In [ ]:
# 模拟因子拥挤分析
# 1. 因子多空组合估值差
print('===== 因子拥挤分析 =====')

for factor_name in factor_orth.columns:
    factor_vals = factor_orth[factor_name].sort_values()
    n = len(factor_vals)
    
    # 多空组合：Top 20% vs Bottom 20%
    n_q = max(1, n // 5)
    top_idx = factor_vals.index[-n_q:]
    bottom_idx = factor_vals.index[:n_q]
    
    # 看这些股票的估值
    top_pe = pd.Series({c: pe_dict.get(c, np.nan) for c in top_idx}).dropna()
    bottom_pe = pd.Series({c: pe_dict.get(c, np.nan) for c in bottom_idx}).dropna()
    
    if len(top_pe) > 0 and len(bottom_pe) > 0:
        # 估值差 = (top组的平均PE倒数) / (bottom组的平均PE倒数) - 1
        top_ep = (1.0 / top_pe).mean()
        bottom_ep = (1.0 / bottom_pe).mean()
        valuation_spread = top_ep / bottom_ep - 1
        print(f'{factor_name}: 多空估值差={valuation_spread:.2%}', end='')
        if abs(valuation_spread) > 0.5:
            print(' ⚠️ 估值差较大，可能存在拥挤')
        else:
            print(' ✅ 估值差正常')

# 2. 因子内相关性分析
print('\n===== 因子内个股相关性 =====')
for factor_name in factor_orth.columns:
    # 取因子暴露最高的10只股票
    factor_vals = factor_orth[factor_name].sort_values(ascending=False)
    top10 = factor_vals.index[:10]
    
    # 计算这10只股票的收益相关性
    avail_stocks = [c for c in top10 if c in price_df.columns]
    if len(avail_stocks) >= 5:
        ret_corr = price_df[avail_stocks].pct_change().dropna().corr()
        avg_corr = (ret_corr.values.sum() - len(avail_stocks)) / (len(avail_stocks) * (len(avail_stocks) - 1))
        print(f'{factor_name} Top10 平均相关性: {avg_corr:.3f}', end='')
        if avg_corr > 0.5:
            print(' ⚠️ 相关性偏高，拥挤信号')
        else:
            print(' ✅ 正常')

## 9. 总结

### 本次我们完成了

| 步骤 | 内容 | 状态 |
|------|------|------|
| 因子构建 | 估值/动量/质量三类因子 | ✅ |
| 因子预处理 | 去极值(MAD) + 标准化 + 正交化 | ✅ |
| IC 分析 | Rank IC / Normal IC 计算与解读 | ✅ |
| IC 衰减 | 多持有期 IC 半衰期分析 | ✅ |
| IR 分析 | 信息比率+年化IR+稳定性 | ✅ |
| 因子稳定性 | t检验 + IC>0比例 | ✅ |
| 因子拥挤 | 估值差 + 相关性分析 | ✅ |

### 验收自检

- [x] 能解释 IC 和 IR 的含义：IC 衡量预测力，IR 衡量稳定性
- [x] 理解因子衰减现象：IC 随持有期递减，半衰期衡量衰减速度
- [x] 知道因子拥挤的风险：估值泡沫、踩踏回撤、因子失效

### 关键概念速查

| 指标 | 公式 | 含义 |
|------|------|------|
| Rank IC | Spearman(Factor, FwdReturn) | 因子排名预测力 |
| IR | mean(IC) / std(IC) | 因子预测稳定性 |
| IC 半衰期 | IC 降至一半的天数 | 因子衰减速度 |
| IC>0 比例 | P(IC > 0) | 因子方向一致性 |

> **下一步**：序号26 — 回测陷阱专题，用本篇构建的因子做回测，识别幸存者偏差、前视偏差等陷阱。